In [1]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Layer

In [9]:
texts = [
    "I love this product",
    "This is amazing",
    "Very happy with the service",
    "Absolutely fantastic",
    "I really like this",

    "I hate this",
    "This is terrible",
    "Worst experience ever",
    "Very bad service",
    "I dislike this product"
]

labels = [1,1,1,1,1, 0,0,0,0,0]
labels = np.array(labels)

In [10]:
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

vocab_size = len(tokenizer.word_index) + 1

sequences = tokenizer.texts_to_sequences(texts)

max_len = max(len(seq) for seq in sequences)

padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

print("Vocab Size:", vocab_size)
print("Max Length:", max_len)
print(padded_sequences)

Vocab Size: 24
Max Length: 5
[[ 3  8  2  4  0]
 [ 2  5  9  0  0]
 [ 6 10 11 12  7]
 [13 14  0  0  0]
 [ 3 15 16  2  0]
 [ 3 17  2  0  0]
 [ 2  5 18  0  0]
 [19 20 21  0  0]
 [ 6 22  7  0  0]
 [ 3 23  2  4  0]]


In [11]:
class Attention(Layer):
    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], 1), initializer="random_normal")
        self.b = self.add_weight(shape=(input_shape[1], 1), initializer="zeros")

    def call(self, inputs):
        score = tf.tanh(tf.matmul(inputs, self.W) + self.b)
        weights = tf.nn.softmax(score, axis=1)

        context = inputs * weights
        context = tf.reduce_sum(context, axis=1)

        return context, weights

In [12]:
inputs = Input(shape=(max_len,))

x = Embedding(vocab_size, 16)(inputs)
lstm_out = LSTM(32, return_sequences=True)(x)

context, attention_weights = Attention()(lstm_out)

output = Dense(1, activation='sigmoid')(context)

model = Model(inputs=inputs, outputs=[output, attention_weights])

model.compile(
    loss=['binary_crossentropy', None],
    optimizer='adam',
    metrics=['accuracy', None]
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)           │ (None, 5)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ embedding_1 (Embedding)              │ (None, 5, 16)               │             384 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 5, 32)               │           6,272 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ attention_1 (Attention)              │ [(None, 32), (None, 5, 1)]  │              37 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,726 (26.27 KB)

 Trainable params: 6,726 (26.27 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# Dummy attention output (needed for training)
dummy_attention = np.zeros((len(padded_sequences), max_len, 1))

model.fit(
    padded_sequences,
    [labels, dummy_attention],
    epochs=100,
    verbose=1
)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - dense_1_accuracy: 0.5000 - loss: 0.6931
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - dense_1_accuracy: 0.5000 - loss: 0.6927
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - dense_1_accuracy: 0.5000 - loss: 0.6923
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - dense_1_accuracy: 0.6000 - loss: 0.6919
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - dense_1_accuracy: 0.6000 - loss: 0.6915
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - dense_1_accuracy: 0.7000 - loss: 0.6911
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - dense_1_accuracy: 0.8000 - loss: 0.6907
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - dense_1_accuracy: 0.8000 - loss: 0.6902
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 171ms/step - dense_1_accuracy: 0.9000 - loss: 0.6897
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step - dense_1_accuracy: 0.9000 - loss: 0.6892
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - dense_1_accuracy: 

In [14]:
test_text = ["I really hate this product"]

seq = tokenizer.texts_to_sequences(test_text)
padded = pad_sequences(seq, maxlen=max_len, padding='post')

pred, attn = model.predict(padded)

print("Prediction:", pred[0][0])
print("Attention:", attn)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 515ms/step
Prediction: 0.84104097
Attention: [[[0.14139567]
  [0.15633911]
  [0.1990045 ]
  [0.24403891]
  [0.25922182]]]


In [16]:
import pickle

# Save model
model.save("model.h5")   # or "model.keras"

# Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("✅ Saved successfully!")

✅ Saved successfully!
